# Deep Research API 简介

## 背景

Deep Research API 让你能够自动化复杂的研究工作流，这些工作流需要对真实世界的信息进行推理、规划和综合。它的设计目标是接收一个高层次的查询，借助能够分解任务、执行网页搜索并综合结果的智能体模型，返回一份结构化且引用丰富的报告。

与 ChatGPT 将这一过程抽象隐藏不同，API 提供了直接的编程访问。当你发送请求时，模型会自主规划子问题，使用网页搜索和代码执行等工具，并生成最终的结构化响应。本 cookbook 将简要介绍 Deep Research API 及其使用方法。

你可以通过 `responses` 端点使用以下模型访问 Deep Research：

- `o3-deep-research-2025-06-26`：针对深度综合和更高质量的输出进行了优化  
- `o4-mini-deep-research-2025-06-26`：更轻量、更快速，非常适合对延迟敏感的用例

## 设置

### 安装依赖
安装最新版本的 OpenAI Python SDK。

In [ ]:
!pip install --upgrade openai

### 身份验证
导入 OpenAI 客户端并使用你的 API 密钥进行初始化。

In [ ]:
from openai import OpenAI
OPENAI_API_KEY="" # YOUR OPENAI_API_KEY
#OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

## 快速开始

我们来看一个 Deep Research API 调用的示例。假设我们在一家医疗金融服务公司工作，任务是就近期用于治疗 2 型糖尿病和肥胖症的药物（尤其是司美格鲁肽）的经济影响撰写一份深度报告。我们的目标是将临床结果、成本效益和区域定价数据综合成一份结构化、有引用支撑的分析，为投资、支付方策略或政策建议提供参考。

首先，我们要：
- 在系统消息中写明我们的角色，并概述希望生成的报告类型。
- 将 `summary` 参数设为 `"auto"`，以获得最佳的可用摘要。如需更详细的报告，可将 `summary` 设为 `"detailed"`。
- 包含必需的 `web_search_preview` 工具，并可选择添加 `code_interpreter`。
- 将 `background` 参数设为 `True`。由于 Deep Research 任务可能需要几分钟才能执行完毕，启用后台模式可以让你异步运行请求，而不必担心超时或其他连接问题。

In [ ]:
system_message = """
You are a professional researcher preparing a structured, data-driven report on behalf of a global health economics team. Your task is to analyze the health question the user poses.

Do:
- Focus on data-rich insights: include specific figures, trends, statistics, and measurable outcomes (e.g., reduction in hospitalization costs, market size, pricing trends, payer adoption).
- When appropriate, summarize data in a way that could be turned into charts or tables, and call this out in the response (e.g., “this would work well as a bar chart comparing per-patient costs across regions”).
- Prioritize reliable, up-to-date sources: peer-reviewed research, health organizations (e.g., WHO, CDC), regulatory agencies, or pharmaceutical earnings reports.
- Include inline citations and return all source metadata.

Be analytical, avoid generalities, and ensure that each section supports data-backed reasoning that could inform healthcare policy or financial modeling.
"""

user_query = "Research the economic impact of semaglutide on global healthcare systems."

response = client.responses.create(
  model="o3-deep-research",
  input=[
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": system_message,
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": user_query,
        }
      ]
    }
  ],
  reasoning={
    "summary": "auto"
  },
  tools=[
    {
      "type": "web_search_preview"
    },
    {
      "type": "code_interpreter",
      "container": {
        "type": "auto",
        "file_ids": []
      }
    }
  ]
)

## 解析响应

Deep Research API 的响应包含结构化的最终答案，以及内联引用、推理步骤摘要和来源元数据。

### 提取最终报告输出

这是该报告的主要文本输出。


In [ ]:
# Access the final report from the response object
print(response.output[-1].content[0].text)

### 访问内联引用和元数据
响应文本中的内联引用带有注解，并链接到对应的来源元数据。每条注解包含：
- start_index 和 end_index：引用所指向的文本字符范围
- title：来源的简短标题
- url：完整的来源 URL

借助这种结构，你可以构建引用列表或参考文献，在下游应用中添加可点击的超链接，并在报告中高亮和追溯有数据支撑的论断。

In [ ]:
annotations = response.output[-1].content[0].annotations
for i, citation in enumerate(annotations):
    print(f"Citation {i+1}:")
    print(f"  Title: {citation.title}")
    print(f"  URL: {citation.url}")
    print(f"  Location: chars {citation.start_index}–{citation.end_index}")


### 检查中间步骤
Deep Research API 还会公开智能体执行的所有中间步骤，包括推理步骤、网页搜索调用和代码执行。你可以用它们来调试、分析或可视化最终答案是如何构建的。
每个中间步骤都存储在 `response.output` 中，`type` 字段表明其类型。

#### 推理步骤
这些是模型在推理子问题时生成的内部摘要或计划。

In [ ]:
# Find the first reasoning step
reasoning = next(item for item in response.output if item.type == "reasoning")
for s in reasoning.summary:
    print(s.text)

#### 网页搜索调用
这些显示了执行了哪些搜索查询，可以帮助你追溯模型检索到了哪些信息。

In [ ]:
# Find the first web search step
search = next(item for item in response.output if item.type == "web_search_call")
print("Query:", search.action["query"])
print("Status:", search.status)

#### 代码执行
如果模型使用了代码解释器（例如用于解析数据或生成图表），这些步骤将以 "code_interpreter_call" 或类似类型出现。

In [ ]:
# Find a code execution step (if any)
code_step = next((item for item in response.output if item.type == "code_interpreter_call"), None)
if code_step:
    print(code_step.input)
    print(code_step.output)
else:
    print("No code execution steps found.")


#### Model Context Protocol (MCP)

假设你希望在 Deep Research 任务中引入自己的内部文档。Deep Research 模型和 Responses API 都支持基于 MCP 的工具，因此你可以扩展它们来查询私有知识库或其他第三方服务。

在下面的示例中，我们配置了一个 MCP 工具，让 Deep Research 可以按需获取你组织内部的司美格鲁肽研究资料。该 MCP 服务器是 OpenAI 文件存储服务的代理，会自动将你上传的文件向量化，以实现高性能检索。

如果你想了解我们是_如何_构建这个简单的 MCP 服务器的，请参阅[这篇相关的 cookbook](https://cookbook.openai.com/examples/deep_research_api/how_to_build_a_deep_research_mcp_server/readme)。

In [ ]:
# system_message includes reference to internal file lookups for MCP.
system_message = """
You are a professional researcher preparing a structured, data-driven report on behalf of a global health economics team. Your task is to analyze the health question the user poses.

Do:
- Focus on data-rich insights: include specific figures, trends, statistics, and measurable outcomes (e.g., reduction in hospitalization costs, market size, pricing trends, payer adoption).
- When appropriate, summarize data in a way that could be turned into charts or tables, and call this out in the response (e.g., “this would work well as a bar chart comparing per-patient costs across regions”).
- Prioritize reliable, up-to-date sources: peer-reviewed research, health organizations (e.g., WHO, CDC), regulatory agencies, or pharmaceutical earnings reports.
- Include an internal file lookup tool to retrieve information from our own internal data sources. If you’ve already retrieved a file, do not call fetch again for that same file. Prioritize inclusion of that data.
- Include inline citations and return all source metadata.

Be analytical, avoid generalities, and ensure that each section supports data-backed reasoning that could inform healthcare policy or financial modeling.
"""

user_query = "Research the economic impact of semaglutide on global healthcare systems."

response = client.responses.create(
  model="o3-deep-research-2025-06-26",
  input=[
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": system_message,
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": user_query,
        }
      ]
    }
  ],
  reasoning={
    "summary": "auto"
  },
  tools=[
    {
      "type": "web_search_preview"
    },
    { # ADD MCP TOOL SUPPORT
      "type": "mcp",
      "server_label": "internal_file_lookup",
      "server_url": "https://<your_mcp_server>/sse/", # Update to the location of *your* MCP server
      "require_approval": "never"
    }
  ]
)


## 查看响应

研究报告的前 100 个字符，随后是引用和 MCP 工具调用。 

In [ ]:
# Grab the full report text once
report_text = response.output[-1].content[0].text

print("REPORT EXCERPT:")
print(report_text[:100])   # first 100 chars
print("--------")

annotations = response.output[-1].content[0].annotations
target_url = "https://platform.openai.com/storage/files"

for citation in annotations:
    if citation.url.startswith(target_url):
        start, end = citation.start_index, citation.end_index

        # extract exactly the cited span
        excerpt       = report_text[start:end]

        # extract up to 100 chars immediately before the citation
        pre_start     = max(0, start - 100)
        preceding_txt = report_text[pre_start:start]

        print("MCP CITATION SAMPLE:")
        print(f"  Title:       {citation.title}")
        print(f"  URL:         {citation.url}")
        print(f"  Location:    chars {start}–{end}")
        print(f"  Preceding:   {preceding_txt!r}")
        print(f"  Excerpt:     {excerpt!r}")
        break

print("--------")


# EXAMPLE MCP CITATION

# REPORT EXCERPT:
# # Introduction
# Semaglutide – a glucagon-like peptide-1 (GLP-1) analogue – has rapidly become a blo
# --------
# MCP CITATION SAMPLE:
#   Title:       Document file-WqbCdYNqNzGuFfCAeWyZfp
#   URL:         https://platform.openai.com/storage/files/file-WqbCdYNqNzGuFfCAeWyZfp
#   Location:    chars 237–331
#   Preceding:   'and obesity due to its potent clinical efficacy (often inducing ~10–15% body weight loss in trials) '
#   Excerpt:     '([platform.openai.com](https://platform.openai.com/storage/files/file-WqbCdYNqNzGuFfCAeWyZfp))'


# print the MCP tool calls
calls = [
    (item.name, item.server_label, item.arguments)
    for item in response.output
    if item.type == "mcp_call" and item.arguments
]
for name, server, args in calls:
    print(f"{name}@{server} → {args}")
print("--------")

## ChatGPT 与 Deep Research API 中的澄清问题
如果你在 ChatGPT 中使用过 Deep Research，可能会注意到它常常在你提交查询后提出后续问题。这是有意为之的：ChatGPT 会使用一个中间模型（如 gpt-4.1）帮助澄清你的意图，并在研究过程开始前收集更多上下文（例如你的偏好、目标或约束）。这一额外步骤有助于系统调整网页搜索，返回更相关、更有针对性的结果。

相比之下，Deep Research API 会跳过这一澄清步骤。作为开发者，你可以自行配置这一处理步骤来改写用户提示词或提出一组澄清问题，因为模型期望一开始就收到完整的提示词，不会要求额外的上下文或补全缺失的信息；它只会根据收到的输入直接开始研究。

要从 API 获得强大且可靠的输出，你可以采用两种方法。
- 使用另一个轻量级模型（例如 gpt-4.1）作为提示词改写器，在将用户查询传递给研究模型之前对其进行扩展或细化。
- 包含所有相关细节：期望的范围、比较对象、指标、地区、首选来源以及预期的输出格式。

这种设置让开发者能够完全控制研究任务的表述方式，但也对输入提示词的质量提出了更高要求。下面是一个通用 rewriting_prompt 的示例，用于更好地引导后续的深度研究查询。

![../../images/intro_dr.png](../../../images/intro_dr.png)

下面是一个改写提示词的示例：

In [ ]:
suggested_rewriting_prompt = """
You will be given a research task by a user. Your job is to produce a set of instructions for a researcher that will complete the task. Do NOT complete the task yourself, just provide instructions on how to complete it.

GUIDELINES:
1. **Maximize Specificity and Detail**
- Include all known user preferences and explicitly list key attributes or dimensions to consider.
- It is of utmost importance that all details from the user are included in the instructions.

2. **Fill in Unstated But Necessary Dimensions as Open-Ended**
- If certain attributes are essential for a meaningful output but the user has not provided them, explicitly state that they are open-ended or default to no specific constraint.

3. **Avoid Unwarranted Assumptions**
- If the user has not provided a particular detail, do not invent one.
- Instead, state the lack of specification and guide the researcher to treat it as flexible or accept all possible options.

4. **Use the First Person**
- Phrase the request from the perspective of the user.

5. **Tables**
- If you determine that including a table will help illustrate, organize, or enhance the information in the research output, you must explicitly request that the researcher provide them.
Examples:
- Product Comparison (Consumer): When comparing different smartphone models, request a table listing each model's features, price, and consumer ratings side-by-side.
- Project Tracking (Work): When outlining project deliverables, create a table showing tasks, deadlines, responsible team members, and status updates.
- Budget Planning (Consumer): When creating a personal or household budget, request a table detailing income sources, monthly expenses, and savings goals.
Competitor Analysis (Work): When evaluating competitor products, request a table with key metrics, such as market share, pricing, and main differentiators.

6. **Headers and Formatting**
- You should include the expected output format in the prompt.
- If the user is asking for content that would be best returned in a structured format (e.g. a report, plan, etc.), ask the researcher to format as a report with the appropriate headers and formatting that ensures clarity and structure.

7. **Language**
- If the user input is in a language other than English, tell the researcher to respond in this language, unless the user query explicitly asks for the response in a different language.

8. **Sources**
- If specific sources should be prioritized, specify them in the prompt.
- For product and travel research, prefer linking directly to official or primary websites (e.g., official brand sites, manufacturer pages, or reputable e-commerce platforms like Amazon for user reviews) rather than aggregator sites or SEO-heavy blogs.
- For academic or scientific queries, prefer linking directly to the original paper or official journal publication rather than survey papers or secondary summaries.
- If the query is in a specific language, prioritize sources published in that language.
"""

In [ ]:
response = client.responses.create(
    instructions=suggested_rewriting_prompt,
    model="gpt-4.1-2025-04-14",
    input="help me plan a trip to france",
)

In [ ]:
new_query = response.output[0].content[0].text
print(new_query)

在这个例子中，用户提交了一个泛泛的或开放式的查询，没有指定旅行日期、目的地偏好、预算、兴趣或同行人员等关键细节；改写提示词对查询进行了改写，因此 Deep Research 会尝试生成一个宽泛且全面的回答，以覆盖常见的使用场景。

虽然这种行为有助于呈现多种多样的选项，但由于模型必须考虑许多可能的场景，它往往会导致内容冗长、延迟更高以及 token 用量增加。对于会触发复杂规划或综合任务的查询（例如多目的地旅行行程、比较研究、产品选择）来说尤其如此。

与其立即执行宽泛的研究计划，不如尝试先用一个更轻量的模型在生成完整答案之前温和地向用户提出澄清问题，然后再使用改写提示词，为模型提供更清晰的输入。

In [ ]:
suggested_clariying_prompt = """"
You will be given a research task by a user. Your job is NOT to complete the task yet, but instead to ask clarifying questions that would help you or another researcher produce a more specific, efficient, and relevant answer.

GUIDELINES:
1. **Maximize Relevance**
- Ask questions that are *directly necessary* to scope the research output.
- Consider what information would change the structure, depth, or direction of the answer.

2. **Surface Missing but Critical Dimensions**
- Identify essential attributes that were not specified in the user’s request (e.g., preferences, time frame, budget, audience).
- Ask about each one *explicitly*, even if it feels obvious or typical.

3. **Do Not Invent Preferences**
- If the user did not mention a preference, *do not assume it*. Ask about it clearly and neutrally.

4. **Use the First Person**
- Phrase your questions from the perspective of the assistant or researcher talking to the user (e.g., “Could you clarify...” or “Do you have a preference for...”)

5. **Use a Bulleted List if Multiple Questions**
- If there are multiple open questions, list them clearly in bullet format for readability.

6. **Avoid Overasking**
- Prioritize the 3–6 questions that would most reduce ambiguity or scope creep. You don’t need to ask *everything*, just the most pivotal unknowns.

7. **Include Examples Where Helpful**
- If asking about preferences (e.g., travel style, report format), briefly list examples to help the user answer.

8. **Format for Conversational Use**
- The output should sound helpful and conversational—not like a form. Aim for a natural tone while still being precise.
"""


response = client.responses.create(
    instructions=suggested_clariying_prompt,
    model="gpt-4.1-2025-04-14",
    input="help me plan a trip to france",
)

new_query = response.output[0].content[0].text
print(new_query)

In [ ]:
user_follow_up = """I'd like to travel in August. I'd like to visit Paria and Nice. I'd like to keep it under $1500 for a 7 day trip without including flights.
I'm going with my friend. we're both in our mid-twenties. i like history, really good french food and wine, and hiking
"""
instructions_for_DR = client.responses.create(
    instructions=suggested_rewriting_prompt,
    model="gpt-4.1-2025-04-14",
    input=user_follow_up,
)
instructions_for_deep_research = instructions_for_DR.output[0].content[0].text
print(instructions_for_deep_research)

In [ ]:
deep_research_call = client.responses.create(
  model="o4-mini-deep-research-2025-06-26",
  input=[
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": instructions_for_deep_research,
        }
      ]
    },
  ],
  reasoning={
    "summary": "auto"
  },
  tools=[
    {
      "type": "web_search_preview"
    },
  ]
)

In [ ]:
# Access the final report from the response object
print(deep_research_call.output[-1].content[0].text)

大功告成！一份为你即将到来的法国之旅量身打造的深度研究报告！

在本 notebook 中，我们探索了如何使用 Deep Research API 自动化复杂的真实世界研究任务，从分析司美格鲁肽的经济影响，到规划一次适合你的法国之旅。当你需要基于真实世界证据、结构化且有引用支撑的答案时，Deep Research 会大放异彩。一些突出的用例包括：
- 产品比较和市场分析
- 竞争情报和战略报告
- 技术文献综述和政策综合


无论你是想构建研究智能体、生成结构化报告，还是将高质量的综合能力集成到工作流中，我们都希望这里的示例能帮助你入门。

下一步？[Deep Research 智能体](https://cookbook.openai.com/examples/deep_research_api/introduction_to_deep_research_api_agents)